In [0]:
spark.read.option("header", True).csv("abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-23/movie.csv").createOrReplaceTempView("v_movie_2")

In [0]:
%sql
SELECT COUNT(1) FROM v_movie_2;

count(1)
1000


In [0]:
spark.read.option("header", True).csv("abfss://bronze@moviehistory310785.dfs.core.windows.net/2024-12-30/movie.csv").createOrReplaceTempView("v_movie_3")

In [0]:
%sql
SELECT COUNT(1) FROM v_movie_3;

count(1)
725


### Ingestion del archivo "movie.csv"

In [0]:
dbutils.widgets.help()

dbutils.widgets provides utilities for working with notebook widgets. You can create
different types of widgets and get their bound value.

For more info about a method, use dbutils.widgets.help("methodName") .
 combobox(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a combobox input widget with a given name, default value and choices dropdown(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a dropdown input widget a with given name, default value and choices get(name: String): String -> Retrieves current value of an input widget getAll: Map -> Retrieves a mapping of all current values of the input widgets getArgument(name: String, optional: String): String -> (DEPRECATED) Equivalent to get multiselect(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a multiselect input widget with a given name, default value and choices remove(name: String): void -> Removes an input widget from the notebook removeAll: void -> Removes all widgets in the notebook text(name: String, defaultValue: String, label: String): void -> Creates a text input widget with a given name and default value

In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
v_environment

'production'

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
bronze_folder_path

'abfss://bronze@moviehistory310785.dfs.core.windows.net'

In [0]:
%run "../Includes/common_functions"

### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
movie_df = spark.read \
    .option("header", True) \
    .csv(f"{bronze_folder_path}/{v_file_date}/movie.csv")

In [0]:
display(movie_df)

movieId,title,budget,homePage,overview,popularity,yearReleaseDate,releaseDate,revenue,durationTime,movieStatus,tagline,voteAverage,voteCount
117942,Girls Gone Dead,500000,http://girlsgonedeadmovie.com/,A group of six ex-high school cheerleaders are stalked by a killer with a medieval war hammer and battle axe during their first Spring Break from college.,1.600171,2012,2012-03-28,625000,104,Released,null,3.50,14
118340,Guardians of the Galaxy,170000000,http://marvel.com/guardians,"Light years from Earth, 26 years after being abducted, Peter Quill finds himself the prime target of a manhunt after discovering an orb wanted by Ronan the Accuser.",481.098624,2014,2014-07-30,773328629,121,Released,All heroes start somewhere.,7.90,9742
118452,"First Love, Last Rites",300000,null,Joey and Sissel are two misfits spending most of their time together talking or having sex. Gradually and slowly their relationships are becoming boring for them.,0.374291,1998,1998-08-07,40542,94,Released,null,3.00,1
118612,After,2000000,null,"When two bus crash survivors awake to discover that they are the only people left in their small town, they must form an unlikely alliance in a race to unravel the truth behind their isolation. As strange events begin to unfold, they start to question whether the town they know so well is really what it seems.",4.596157,2012,2012-08-27,2600000,90,Released,null,5.60,63
118957,Bait,30000000,null,A freak tsunami traps shoppers at a coastal Australian supermarket inside the building ... along with a 12-foot great white shark.,9.780588,2012,2012-09-05,37500000,93,Released,Cleanup on aisle 7.,5.30,191
119283,Parker,35000000,null,"A thief with a unique code of professional ethics is double-crossed by his crew and left for dead. Assuming a new disguise and forming an unlikely alliance with a woman on the inside, he looks to hijack the score of the crew's latest heist.",28.670477,2013,2013-01-23,46216641,118,Released,"To get away clean, you have to play dirty.",5.70,1455
119450,Dawn of the Planet of the Apes,170000000,http://www.dawnofapes.com/,"A group of scientists in San Francisco struggle to stay alive in the aftermath of a plague that is wiping out humanity, while Caesar tries to maintain dominance over his community of intelligent apes.",243.791743,2014,2014-06-26,710644566,130,Released,One last chance for peace.,7.30,4410
119458,$upercapitalist,2000000,http://supercapitalist.net/,A maverick New York hedge fund trader with uncanny analytic abilities moves to Hong Kong and orchestrates a mega-deal that swiftly escalates beyond his control.,0.174311,2012,2012-08-10,2600000,103,Released,Money for Life,3.50,2
119657,El Rey de Najayo,2000000,null,"The dramatic story of Julian, a Dominican drug lord who despite his confinement in prison, was still able to maintain Dominican society in a state of suspense, for over 13 years. At the early age of 12, he witnessed the death of his father at the hands of local military authorities, during a well-meant attempt to hand-over a package of drugs he had incidentally found at sea while fishing. As a result of this experience, Julian develops a thirst for revenge that leads him to kill all those involved in his father's death. In the process he becomes a major drug lord and a very powerful headman within Dominican society.",0.080105,2012,2012-03-01,2600000,101,Released,null,0.00,0
120467,The Grand Budapest Hotel,30000000,null,"The Grand Budapest Hotel tells of a legendary concierge at a famous European hotel between the wars and his friendship with a young employee who becomes his trusted protégé. The story involves the theft and recovery of a priceless Renaissance painting, the battle for an enormous family fortune and the slow and then sudden upheavals that transformed Europe during the first half of the 20th century.",74.417456,2014,2014-02-26,174600318,99,Released,A perfect holiday without leaving home.,8.00,4519


In [0]:
movie_df.printSchema()

root
 |-- movieId: string (nullable = true)
 |-- title: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- homePage: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- yearReleaseDate: string (nullable = true)
 |-- releaseDate: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- durationTime: string (nullable = true)
 |-- movieStatus: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- voteAverage: string (nullable = true)
 |-- voteCount: string (nullable = true)



In [0]:
display(movie_df.describe())

summary,movieId,title,budget,homePage,overview,popularity,yearReleaseDate,releaseDate,revenue,durationTime,movieStatus,tagline,voteAverage,voteCount
count,725,725,725,308,722,725,725,725,725,725,725,525,723,722
mean,243797.31862068965,1982.0,2.3835064707586206E7,null,null,28.881584015965178,1976.8483123874648,1201.454686090909,7.687520299974386E7,1371097.004800721,2.5459956309888564E7,1750328.3550628002,446559.2628040059,29048.438640762302
stddev,73971.93123230412,null,4.042001723669771E7,null,null,61.159821075256666,267.5819985756023,999.473410936445,1.7541675739552444E8,1.6489210428140983E7,9.775631595335196E7,6439874.488290873,7155682.78711931,663654.0954207863
min,117942,#Horror,100000,http://2016themovie.com/,"""""""Selma",Adam,"Ahmad Shah. Only one member of the team survived.""",a loving wife and daughter,Barack Obama. What they didn't know is that Obama is a man with a past,and along the way get a first look at the new menace from Episode III,Billy hits rock bottom,and where he intends to take America and the world. Immersed in exotic locales across four continents,"best selling author Dinesh DSouza races against time to find answers to Obama's past and reveal where America will be in 2016.""",a former fighter who trains the city's toughest amateur boxers. With his future on the line
max,459488,Évolution,99000000,https://www.uphe.com/movies/legend-2015,"“The Perfect Wave” is the true story of Ian McCormack who grew up surfing the waters of New Zealand. Wanting to dive deeper, Ian sets out on a journey with his best friend that will change his life as they chase the perfect wave.","\"""" is shot by an unknown assailant",60.246978,9.799357,"\"""" and weave an unforgettable",99,Released,"two brothers, four women and the search for Magnetic Perfection",Where Pride Began,Witness the rise of a legend


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, DateType

In [0]:
movie_schema = StructType(fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
] )

In [0]:
movie_df = spark.read \
    .option("header", True) \
    .schema(movie_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/movie.csv")

### Paso 2 - Seleccionar solo las columnas "requeridas"

In [0]:
from pyspark.sql.functions import col

In [0]:
movie_selected_df = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"), col("voteCount"))

### Paso 3 - Cambiar el nombre de las columnas segun lo "requerido"

In [0]:
movie_renamed_df = movie_selected_df \
                   .withColumnRenamed("movieId", "movie_Id") \
                   .withColumnRenamed("yearReleaseDate", "year_Release_Date") \
                   .withColumnRenamed("releaseDate", "release_Date") \
                   .withColumnRenamed("durationTime", "duration_Time") \
                   .withColumnRenamed("voteAverage", "vote_Average") \
                   .withColumnRenamed("voteCount", "vote_Count")

In [0]:
movie_renamed_df = movie_selected_df \
                   .withColumnsRenamed({"movieId": "movie_Id", "yearReleaseDate": "year_Release_Date", "releaseDate": "release_Date", "durationTime": "duration_Time", "voteAverage": "vote_Average", "voteCount": "vote_Count"})

#### Paso 4 - Agregar la columna "ingestion_date" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_final_df = movie_renamed_df \
                  .withColumn("ingestion_date", current_timestamp()) \
                  .withColumn("env", lit(v_environment)) \
                  .withColumn("file_date", lit(v_file_date)) 

In [0]:
movies_final_df = movie_renamed_df \
                  .withColumns({"ingestion_date": current_timestamp(), "env": lit(v_environment)}) \
                  .withColumn("file_date", lit(v_file_date))

In [0]:
movies_final_df = add_ingestion_date(movie_renamed_df) \
                  .withColumn("env", lit(v_environment)) \
                  .withColumn("file_date", lit(v_file_date))

#### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
# overwrite_partition("movie_silver", "movies", "file_date", v_file_date)

In [0]:
merge_delta_lake(movies_final_df, "movie_silver", "movies", "movie_Id", "file_date")

In [0]:
# movies_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies")

In [0]:
%sql
SELECT file_date, COUNT(1) 
FROM movie_silver.movies
GROUP BY file_date;

file_date,count(1)
2024-12-16,3000
2024-12-23,1000
2024-12-30,725


In [0]:
df = spark.read.table("movie_silver.movies")

In [0]:
display(df)

movie_Id,title,budget,popularity,year_Release_Date,release_Date,revenue,duration_Time,vote_Average,vote_Count,ingestion_date,env,file_date
5,Four Rooms,4000000.0,22.87623,1995,1995-12-09,4300000.0,98,6.5,530,2026-09-11T03:20:16.841432Z,production,2024-12-16
11,Star Wars,1.1E7,126.393695,1977,1977-05-25,7.75398007E8,121,8.1,6624,2026-09-11T03:20:16.841432Z,production,2024-12-16
12,Finding Nemo,9.4E7,85.688789,2003,2003-05-30,9.40335536E8,100,7.6,6122,2026-09-11T03:20:16.841432Z,production,2024-12-16
13,Forrest Gump,5.5E7,138.133331,1994,1994-07-06,6.77945399E8,142,8.2,7927,2026-09-11T03:20:16.841432Z,production,2024-12-16
14,American Beauty,1.5E7,80.878605,1999,1999-09-15,3.56296601E8,122,7.9,3313,2026-09-11T03:20:16.841432Z,production,2024-12-16
16,Dancer in the Dark,1.28E7,22.022228,2000,2000-05-17,4.0031879E7,140,7.6,377,2026-09-11T03:20:16.841432Z,production,2024-12-16
18,The Fifth Element,9.0E7,109.528572,1997,1997-05-07,2.6392018E8,126,7.3,3885,2026-09-11T03:20:16.841432Z,production,2024-12-16
19,Metropolis,9.262E7,32.351527,1927,1927-01-10,650422.0,153,8.0,657,2026-09-11T03:20:16.841432Z,production,2024-12-16
20,My Life Without Me,2000000.0,7.958831,2003,2003-03-07,2600000.0,106,7.2,77,2026-09-11T03:20:16.841432Z,production,2024-12-16
22,Pirates of the Caribbean: The Curse of the Black Pearl,1.4E8,271.972889,2003,2003-07-09,6.55011224E8,143,7.5,6985,2026-09-11T03:20:16.841432Z,production,2024-12-16


In [0]:
display(spark.read.table("movie_silver.movies"))

movie_Id,title,budget,popularity,year_Release_Date,release_Date,revenue,duration_Time,vote_Average,vote_Count,ingestion_date,env,file_date
5,Four Rooms,4000000.0,22.87623,1995,1995-12-09,4300000.0,98,6.5,530,2026-09-11T03:20:16.841432Z,production,2024-12-16
11,Star Wars,1.1E7,126.393695,1977,1977-05-25,7.75398007E8,121,8.1,6624,2026-09-11T03:20:16.841432Z,production,2024-12-16
12,Finding Nemo,9.4E7,85.688789,2003,2003-05-30,9.40335536E8,100,7.6,6122,2026-09-11T03:20:16.841432Z,production,2024-12-16
13,Forrest Gump,5.5E7,138.133331,1994,1994-07-06,6.77945399E8,142,8.2,7927,2026-09-11T03:20:16.841432Z,production,2024-12-16
14,American Beauty,1.5E7,80.878605,1999,1999-09-15,3.56296601E8,122,7.9,3313,2026-09-11T03:20:16.841432Z,production,2024-12-16
16,Dancer in the Dark,1.28E7,22.022228,2000,2000-05-17,4.0031879E7,140,7.6,377,2026-09-11T03:20:16.841432Z,production,2024-12-16
18,The Fifth Element,9.0E7,109.528572,1997,1997-05-07,2.6392018E8,126,7.3,3885,2026-09-11T03:20:16.841432Z,production,2024-12-16
19,Metropolis,9.262E7,32.351527,1927,1927-01-10,650422.0,153,8.0,657,2026-09-11T03:20:16.841432Z,production,2024-12-16
20,My Life Without Me,2000000.0,7.958831,2003,2003-03-07,2600000.0,106,7.2,77,2026-09-11T03:20:16.841432Z,production,2024-12-16
22,Pirates of the Caribbean: The Curse of the Black Pearl,1.4E8,271.972889,2003,2003-07-09,6.55011224E8,143,7.5,6985,2026-09-11T03:20:16.841432Z,production,2024-12-16


In [0]:
%sql
DESCRIBE EXTENDED movie_silver.movies;

col_name,data_type,comment
movie_Id,int,null
title,string,null
budget,double,null
popularity,double,null
year_Release_Date,int,null
release_Date,date,null
revenue,double,null
duration_Time,int,null
vote_Average,double,null
vote_Count,int,null


In [0]:
dbutils.notebook.exit("Success")